# 🌙 Lunar Habitat AI — XGBoost Training Pipeline
### MitWPU Hackathon | Google Colab Edition

**Pipeline Steps:**
1. Install dependencies
2. Load the 23 real lunar site scientific dataset
3. Download & sample the 8GB LOLA Global DEM for real topographic data
4. Engineer features & build the training dataset (synthetic augmentation)
5. Train XGBoost model to predict Habitat Suitability Score
6. Run inference on all 23 real sites
7. Compute SHAP feature importance
8. Export `ai_predictions.json` for React frontend integration

---
> **Instructions:** Click `Runtime > Run all` to execute the full pipeline. Download `ai_predictions.json` at the end.

## Step 1: Install Dependencies

In [ ]:
!pip install -q xgboost shap optuna rasterio tifffile numpy pandas scikit-learn matplotlib seaborn joblib
print('✅ All dependencies installed successfully!')

## Step 2: Load the 23-Site Scientific Dataset
Paste the content of your `src/data/lunar_scientific_dataset.json` below, OR upload the file to Colab and load it from disk.

In [ ]:
import json
import numpy as np
import pandas as pd

# ── Dataset is embedded directly — no upload needed! ─────────────────────
LUNAR_DATASET_JSON = '[\n  {\n    "id": "site-shackleton",\n    "node_id": "01_Shackleton_Crater",\n    "code": "Shackleton",\n    "name": "Shackleton Crater Rim — Peak of Eternal Light",\n    "coordinates": {\n      "latitude": -89.28,\n      "longitude": 15.4,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-01-A"\n    },\n    "terrain_dem": {\n      "elevation_m": 4120,\n      "slope_deg": 4.2,\n      "roughness_rms_m": 0.85,\n      "crater_diameter_km": 21.0,\n      "rim_depth_m": 4200,\n      "landing_corridor_rating": "Optimal Crest Line",\n      "accessibility_index_100": 82\n    },\n    "water_ice": {\n      "ice_probability_pct": 89.0,\n      "hydrogen_content_ppm": 1450,\n      "radar_cpr": 0.78,\n      "spectroscopy_band_3um_depth": 0.082,\n      "distance_to_psr_m": 350,\n      "estimated_ice_depth_m": 1.2,\n      "psr_name": "Shackleton Interior Basin"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 95.2,\n      "max_continuous_light_days": 180,\n      "max_continuous_dark_days": 3.5,\n      "avg_solar_elevation_deg": 1.45,\n      "seasonal_variance_pct": 4.8\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 280,\n      "spe_hazard_tier": "Moderate (Elevated Ridge)",\n      "dose_rate_usv_h": 32.0,\n      "solar_cycle_phase": "Cycle 25 Maximum Modulation",\n      "terrain_shielding_factor_pct": 84.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Anorthositic Impact Melt Breccia (Pre-Nectarian)",\n      "crater_boundary": "South Pole-Aitken (SPA) Basin Margin",\n      "earth_direct_los_pct": 98.4,\n      "relay_satellite_required": false,\n      "near_or_far_side": "South Polar Axis"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "High-Yield Volatiles / Permanent Light Ridge",\n      "mission_ground_truth_reference": "LRO LOLA / Chandrayaan-1 M3 / Artemis III Target",\n      "mcda_suitability_score": 94.2,\n      "ai_confidence_pct": 94.0,\n      "suitability_tier": "HIGHLY SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 180,\n      "temp_max_k": 220,\n      "diurnal_temperature_swing_k": 40\n    }\n  },\n  {\n    "id": "site-malapert",\n    "node_id": "02_Mons_Malapert",\n    "code": "Malapert",\n    "name": "Mons Malapert (Malapert Mountain Plateau)",\n    "coordinates": {\n      "latitude": -85.99,\n      "longitude": 12.9,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-02-B"\n    },\n    "terrain_dem": {\n      "elevation_m": 5100,\n      "slope_deg": 6.1,\n      "roughness_rms_m": 1.1,\n      "crater_diameter_km": 69.0,\n      "rim_depth_m": 5100,\n      "landing_corridor_rating": "Expansive Plateau Shelf",\n      "accessibility_index_100": 86\n    },\n    "water_ice": {\n      "ice_probability_pct": 83.0,\n      "hydrogen_content_ppm": 980,\n      "radar_cpr": 0.62,\n      "spectroscopy_band_3um_depth": 0.054,\n      "distance_to_psr_m": 1100,\n      "estimated_ice_depth_m": 1.8,\n      "psr_name": "Malapert South Depressions"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 93.6,\n      "max_continuous_light_days": 150,\n      "max_continuous_dark_days": 5.0,\n      "avg_solar_elevation_deg": 1.8,\n      "seasonal_variance_pct": 6.4\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 290,\n      "spe_hazard_tier": "Moderate-High (High Massif Exposure)",\n      "dose_rate_usv_h": 33.1,\n      "solar_cycle_phase": "Cycle 25 Maximum Modulation",\n      "terrain_shielding_factor_pct": 88.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Highland Anorthosite Crustal Massif",\n      "crater_boundary": "Malapert Crater West Flank",\n      "earth_direct_los_pct": 99.6,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Near Side / Polar Rim"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Continuous Comms Relay / Surface Basecamp",\n      "mission_ground_truth_reference": "IM-1 Odysseus / LRO Altimetry Survey",\n      "mcda_suitability_score": 91.8,\n      "ai_confidence_pct": 91.0,\n      "suitability_tier": "HIGHLY SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 195,\n      "temp_max_k": 235,\n      "diurnal_temperature_swing_k": 40\n    }\n  },\n  {\n    "id": "site-faustini",\n    "node_id": "03_Faustini_Rim_A",\n    "code": "Faustini A",\n    "name": "Faustini Crater Rim — Ridge A",\n    "coordinates": {\n      "latitude": -87.15,\n      "longitude": 77.0,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-03-C"\n    },\n    "terrain_dem": {\n      "elevation_m": 2450,\n      "slope_deg": 8.5,\n      "roughness_rms_m": 1.45,\n      "crater_diameter_km": 39.0,\n      "rim_depth_m": 3100,\n      "landing_corridor_rating": "Rugged Terraced Rim",\n      "accessibility_index_100": 76\n    },\n    "water_ice": {\n      "ice_probability_pct": 95.0,\n      "hydrogen_content_ppm": 2100,\n      "radar_cpr": 0.92,\n      "spectroscopy_band_3um_depth": 0.115,\n      "distance_to_psr_m": 140,\n      "estimated_ice_depth_m": 0.6,\n      "psr_name": "Faustini Ultra-Cold Trap (40K)"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 77.5,\n      "max_continuous_light_days": 90,\n      "max_continuous_dark_days": 18.0,\n      "avg_solar_elevation_deg": 1.1,\n      "seasonal_variance_pct": 14.5\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 335,\n      "spe_hazard_tier": "Low-Moderate (Crater Wall Shielding)",\n      "dose_rate_usv_h": 38.2,\n      "solar_cycle_phase": "Cycle 25 Maximum Modulation",\n      "terrain_shielding_factor_pct": 80.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Ilmenite-Rich Polar Impact Regolith",\n      "crater_boundary": "Faustini-Shoemaker Inter-Crater Ridge",\n      "earth_direct_los_pct": 86.2,\n      "relay_satellite_required": true,\n      "near_or_far_side": "Far Side / Polar Margin"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Industrial Cryogenic Mining / Volatiles Prospect",\n      "mission_ground_truth_reference": "NASA Diviner Cold Trap / LEND Spectrometry",\n      "mcda_suitability_score": 83.4,\n      "ai_confidence_pct": 85.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 110,\n      "temp_max_k": 200,\n      "diurnal_temperature_swing_k": 90\n    }\n  },\n  {\n    "id": "site-connecting-ridge",\n    "node_id": "04_Connecting_Ridge",\n    "code": "Connecting Ridge",\n    "name": "Connecting Ridge (Shackleton-de Gerlache)",\n    "coordinates": {\n      "latitude": -88.6,\n      "longitude": -31.7,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-04-D"\n    },\n    "terrain_dem": {\n      "elevation_m": 3850,\n      "slope_deg": 3.8,\n      "roughness_rms_m": 0.7,\n      "crater_diameter_km": 15.0,\n      "rim_depth_m": 2200,\n      "landing_corridor_rating": "Wide Corridors (<4 deg Slope)",\n      "accessibility_index_100": 92\n    },\n    "water_ice": {\n      "ice_probability_pct": 86.0,\n      "hydrogen_content_ppm": 1250,\n      "radar_cpr": 0.71,\n      "spectroscopy_band_3um_depth": 0.075,\n      "distance_to_psr_m": 580,\n      "estimated_ice_depth_m": 1.4,\n      "psr_name": "de Gerlache Northern Shadow Corridor"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 89.5,\n      "max_continuous_light_days": 135,\n      "max_continuous_dark_days": 8.0,\n      "avg_solar_elevation_deg": 1.35,\n      "seasonal_variance_pct": 8.2\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 305,\n      "spe_hazard_tier": "Moderate",\n      "dose_rate_usv_h": 34.8,\n      "solar_cycle_phase": "Cycle 25 Maximum Modulation",\n      "terrain_shielding_factor_pct": 87.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Consolidated Sintered Regolith Saddle",\n      "crater_boundary": "Shackleton / de Gerlache Boundary",\n      "earth_direct_los_pct": 93.5,\n      "relay_satellite_required": false,\n      "near_or_far_side": "South Polar Ridge"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Primary Human Lander Touchdown Corridor",\n      "mission_ground_truth_reference": "NASA HLS Designated / LUPEX Candidate",\n      "mcda_suitability_score": 89.6,\n      "ai_confidence_pct": 89.0,\n      "suitability_tier": "HIGHLY SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 160,\n      "temp_max_k": 220,\n      "diurnal_temperature_swing_k": 60\n    }\n  },\n  {\n    "id": "site-de-gerlache",\n    "node_id": "05_de_Gerlache_Rim",\n    "code": "de Gerlache",\n    "name": "de Gerlache Rim — Mons Peak Alpha",\n    "coordinates": {\n      "latitude": -85.9,\n      "longitude": 76.3,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-05-E"\n    },\n    "terrain_dem": {\n      "elevation_m": 2900,\n      "slope_deg": 7.2,\n      "roughness_rms_m": 1.25,\n      "crater_diameter_km": 32.4,\n      "rim_depth_m": 2900,\n      "landing_corridor_rating": "Terraced Rim Crest",\n      "accessibility_index_100": 80\n    },\n    "water_ice": {\n      "ice_probability_pct": 91.0,\n      "hydrogen_content_ppm": 1650,\n      "radar_cpr": 0.82,\n      "spectroscopy_band_3um_depth": 0.092,\n      "distance_to_psr_m": 210,\n      "estimated_ice_depth_m": 0.9,\n      "psr_name": "de Gerlache Deep Floor Trap"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 83.5,\n      "max_continuous_light_days": 110,\n      "max_continuous_dark_days": 12.0,\n      "avg_solar_elevation_deg": 1.2,\n      "seasonal_variance_pct": 11.2\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 320,\n      "spe_hazard_tier": "Moderate",\n      "dose_rate_usv_h": 36.5,\n      "solar_cycle_phase": "Cycle 25 Maximum Modulation",\n      "terrain_shielding_factor_pct": 82.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Solid Anorthosite Bedrock / Impact Ejecta",\n      "crater_boundary": "de Gerlache Northern Crater Wall",\n      "earth_direct_los_pct": 89.2,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Near Side / Polar Margin"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "High-Yield Volatiles / Nuclear ISRU Pilot",\n      "mission_ground_truth_reference": "LRO Neutron Spectrometer & LOLA",\n      "mcda_suitability_score": 84.1,\n      "ai_confidence_pct": 86.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 140,\n      "temp_max_k": 210,\n      "diurnal_temperature_swing_k": 70\n    }\n  },\n  {\n    "id": "site-haworth",\n    "node_id": "06_Haworth_Crater_Rim",\n    "code": "Haworth",\n    "name": "Haworth Crater Rim North",\n    "coordinates": {\n      "latitude": -87.4,\n      "longitude": -5.1,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-06-F"\n    },\n    "terrain_dem": {\n      "elevation_m": 3100,\n      "slope_deg": 5.4,\n      "roughness_rms_m": 0.95,\n      "crater_diameter_km": 51.4,\n      "rim_depth_m": 3800,\n      "landing_corridor_rating": "Even Highland Ridge",\n      "accessibility_index_100": 85\n    },\n    "water_ice": {\n      "ice_probability_pct": 91.0,\n      "hydrogen_content_ppm": 1780,\n      "radar_cpr": 0.85,\n      "spectroscopy_band_3um_depth": 0.098,\n      "distance_to_psr_m": 280,\n      "estimated_ice_depth_m": 0.8,\n      "psr_name": "Haworth Deep Shadow Floor (38K)"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 82.8,\n      "max_continuous_light_days": 105,\n      "max_continuous_dark_days": 14.0,\n      "avg_solar_elevation_deg": 1.25,\n      "seasonal_variance_pct": 12.0\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 315,\n      "spe_hazard_tier": "Moderate",\n      "dose_rate_usv_h": 35.9,\n      "solar_cycle_phase": "Cycle 25 Maximum Modulation",\n      "terrain_shielding_factor_pct": 83.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Inter-Crater Highland Ejecta Blanket",\n      "crater_boundary": "Haworth / Shoemaker Divide",\n      "earth_direct_los_pct": 91.8,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Near Side Polar Rim"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Multi-Basin Geological Science Station",\n      "mission_ground_truth_reference": "NASA Diviner Cryogenic Benchmark",\n      "mcda_suitability_score": 85.8,\n      "ai_confidence_pct": 87.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 130,\n      "temp_max_k": 215,\n      "diurnal_temperature_swing_k": 85\n    }\n  },\n  {\n    "id": "site-mons-mouton",\n    "node_id": "07_Mons_Mouton",\n    "code": "Mons Mouton",\n    "name": "Mons Mouton (Leibnitz Beta Plateau)",\n    "coordinates": {\n      "latitude": -85.1,\n      "longitude": 31.5,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-07-G"\n    },\n    "terrain_dem": {\n      "elevation_m": 5900,\n      "slope_deg": 2.9,\n      "roughness_rms_m": 0.55,\n      "crater_diameter_km": 120.0,\n      "rim_depth_m": 5900,\n      "landing_corridor_rating": "Broad Tableland (15+ sq km)",\n      "accessibility_index_100": 93\n    },\n    "water_ice": {\n      "ice_probability_pct": 81.0,\n      "hydrogen_content_ppm": 890,\n      "radar_cpr": 0.58,\n      "spectroscopy_band_3um_depth": 0.048,\n      "distance_to_psr_m": 1650,\n      "estimated_ice_depth_m": 2.1,\n      "psr_name": "Leibnitz Beta Flank Shadow Traps"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 93.8,\n      "max_continuous_light_days": 160,\n      "max_continuous_dark_days": 4.5,\n      "avg_solar_elevation_deg": 1.95,\n      "seasonal_variance_pct": 5.9\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 275,\n      "spe_hazard_tier": "Moderate-High",\n      "dose_rate_usv_h": 31.4,\n      "solar_cycle_phase": "Cycle 25 Maximum Modulation",\n      "terrain_shielding_factor_pct": 91.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Pre-Nectarian Anorthositic Crustal Massive",\n      "crater_boundary": "Leibnitz Crater Rim Complex",\n      "earth_direct_los_pct": 99.3,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Near Side Polar Highland"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Civil Spaceport / Long-Term Basecamp Complex",\n      "mission_ground_truth_reference": "NASA VIPER Primary Science Target",\n      "mcda_suitability_score": 92.6,\n      "ai_confidence_pct": 95.0,\n      "suitability_tier": "HIGHLY SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 200,\n      "temp_max_k": 240,\n      "diurnal_temperature_swing_k": 40\n    }\n  },\n  {\n    "id": "site-nobile",\n    "node_id": "08_Nobile_Crater_Rim",\n    "code": "Nobile",\n    "name": "Nobile Crater Rim — Artemis Base Alpha",\n    "coordinates": {\n      "latitude": -85.2,\n      "longitude": 53.5,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-08-H"\n    },\n    "terrain_dem": {\n      "elevation_m": 3400,\n      "slope_deg": 4.8,\n      "roughness_rms_m": 0.88,\n      "crater_diameter_km": 79.2,\n      "rim_depth_m": 3700,\n      "landing_corridor_rating": "Rolling Highland Slopes",\n      "accessibility_index_100": 89\n    },\n    "water_ice": {\n      "ice_probability_pct": 88.0,\n      "hydrogen_content_ppm": 1380,\n      "radar_cpr": 0.74,\n      "spectroscopy_band_3um_depth": 0.08,\n      "distance_to_psr_m": 490,\n      "estimated_ice_depth_m": 1.1,\n      "psr_name": "Nobile Crater Shadow Pockets"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 86.8,\n      "max_continuous_light_days": 125,\n      "max_continuous_dark_days": 9.5,\n      "avg_solar_elevation_deg": 1.5,\n      "seasonal_variance_pct": 9.4\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 300,\n      "spe_hazard_tier": "Moderate",\n      "dose_rate_usv_h": 34.2,\n      "solar_cycle_phase": "Cycle 25 Maximum Modulation",\n      "terrain_shielding_factor_pct": 85.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Volatile-Rich Sinuous Ejecta Blanket",\n      "crater_boundary": "Nobile Crater Rim Crest",\n      "earth_direct_los_pct": 94.8,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Near Side / Polar Margin"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Artemis Base Camp Core Habitation Zone",\n      "mission_ground_truth_reference": "VIPER Rover Exploration Zone",\n      "mcda_suitability_score": 87.5,\n      "ai_confidence_pct": 89.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 165,\n      "temp_max_k": 225,\n      "diurnal_temperature_swing_k": 60\n    }\n  },\n  {\n    "id": "site-i",\n    "node_id": "09_Amundsen_Crater",\n    "code": "Amundsen",\n    "name": "Amundsen Crater Floor & Rim (Site I)",\n    "coordinates": {\n      "latitude": -84.5,\n      "longitude": 82.8,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-09-I"\n    },\n    "terrain_dem": {\n      "elevation_m": 1800,\n      "slope_deg": 6.8,\n      "roughness_rms_m": 1.3,\n      "crater_diameter_km": 105.0,\n      "rim_depth_m": 4800,\n      "landing_corridor_rating": "Terraced Wall / Peak Complex",\n      "accessibility_index_100": 78\n    },\n    "water_ice": {\n      "ice_probability_pct": 89.0,\n      "hydrogen_content_ppm": 1520,\n      "radar_cpr": 0.79,\n      "spectroscopy_band_3um_depth": 0.088,\n      "distance_to_psr_m": 380,\n      "estimated_ice_depth_m": 1.0,\n      "psr_name": "Amundsen Vast Central Shadow Basin"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 75.2,\n      "max_continuous_light_days": 85,\n      "max_continuous_dark_days": 21.0,\n      "avg_solar_elevation_deg": 1.15,\n      "seasonal_variance_pct": 16.5\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 330,\n      "spe_hazard_tier": "Low-Moderate (Basin Wall Berms)",\n      "dose_rate_usv_h": 37.6,\n      "solar_cycle_phase": "Cycle 25 Maximum Modulation",\n      "terrain_shielding_factor_pct": 78.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Impact Melt Sheet with Central Uplift",\n      "crater_boundary": "Amundsen Crater Interior Basin",\n      "earth_direct_los_pct": 82.5,\n      "relay_satellite_required": true,\n      "near_or_far_side": "Far Side / Polar Limb"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Deep Catchment Geological & Organics Laboratory",\n      "mission_ground_truth_reference": "LRO LAMP Spectral Survey",\n      "mcda_suitability_score": 80.2,\n      "ai_confidence_pct": 82.0,\n      "suitability_tier": "MODERATE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 120,\n      "temp_max_k": 210,\n      "diurnal_temperature_swing_k": 90\n    }\n  },\n  {\n    "id": "site-j",\n    "node_id": "10_Marius_Hills_Lava_Tube",\n    "code": "Marius Hills",\n    "name": "Marius Hills Lava Tube Skylight (Site J)",\n    "coordinates": {\n      "latitude": 14.2,\n      "longitude": -56.7,\n      "hemisphere": "Oceanus Procellarum",\n      "grid_cell": "EQ-10-J"\n    },\n    "terrain_dem": {\n      "elevation_m": -1200,\n      "slope_deg": 12.0,\n      "roughness_rms_m": 2.1,\n      "crater_diameter_km": 0.065,\n      "rim_depth_m": 50,\n      "landing_corridor_rating": "Volcanic Dome Plain (Skylight Shaft)",\n      "accessibility_index_100": 74\n    },\n    "water_ice": {\n      "ice_probability_pct": 46.0,\n      "hydrogen_content_ppm": 120,\n      "radar_cpr": 0.3,\n      "spectroscopy_band_3um_depth": 0.012,\n      "distance_to_psr_m": 45000,\n      "estimated_ice_depth_m": 0.0,\n      "psr_name": "None (Subterranean Void)"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 52.0,\n      "max_continuous_light_days": 14,\n      "max_continuous_dark_days": 14.0,\n      "avg_solar_elevation_deg": 75.8,\n      "seasonal_variance_pct": 1.2\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 15,\n      "spe_hazard_tier": "Zero (Complete Basalt Overburden Shielding)",\n      "dose_rate_usv_h": 1.7,\n      "solar_cycle_phase": "Protected from Solar Variability",\n      "terrain_shielding_factor_pct": 100.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Sinuous Rille / Subsurface Basalt Lava Tube",\n      "crater_boundary": "Marius Volcanic Complex",\n      "earth_direct_los_pct": 82.0,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Near Side / Western Oceanus"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Natural Radiation Shield / Subterranean Master Colony",\n      "mission_ground_truth_reference": "SELENE (Kaguya) Radar Sounder & LROC Skylight",\n      "mcda_suitability_score": 86.7,\n      "ai_confidence_pct": 88.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 253,\n      "temp_max_k": 256,\n      "diurnal_temperature_swing_k": 3\n    }\n  },\n  {\n    "id": "site-k",\n    "node_id": "11_Cabeus_Crater",\n    "code": "Cabeus",\n    "name": "Cabeus Crater (LCROSS Ground Zero)",\n    "coordinates": {\n      "latitude": -84.9,\n      "longitude": -35.5,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-11-K"\n    },\n    "terrain_dem": {\n      "elevation_m": 2100,\n      "slope_deg": 6.9,\n      "roughness_rms_m": 1.2,\n      "crater_diameter_km": 100.0,\n      "rim_depth_m": 4100,\n      "landing_corridor_rating": "Cryogenic Basin Margin",\n      "accessibility_index_100": 77\n    },\n    "water_ice": {\n      "ice_probability_pct": 97.0,\n      "hydrogen_content_ppm": 2400,\n      "radar_cpr": 0.95,\n      "spectroscopy_band_3um_depth": 0.14,\n      "distance_to_psr_m": 90,\n      "estimated_ice_depth_m": 0.4,\n      "psr_name": "Cabeus LCROSS Impact Plume Site"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 73.5,\n      "max_continuous_light_days": 80,\n      "max_continuous_dark_days": 24.0,\n      "avg_solar_elevation_deg": 1.1,\n      "seasonal_variance_pct": 18.0\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 335,\n      "spe_hazard_tier": "Moderate",\n      "dose_rate_usv_h": 38.2,\n      "solar_cycle_phase": "Cycle 25 Maximum Modulation",\n      "terrain_shielding_factor_pct": 80.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Cryogenic Volatiles / Hydrocarbon Rich Regolith",\n      "crater_boundary": "Cabeus Crater Floor Trap",\n      "earth_direct_los_pct": 86.8,\n      "relay_satellite_required": true,\n      "near_or_far_side": "Near Side Polar Limb"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Confirmed Water Ice Ground Zero (5.6% by Mass)",\n      "mission_ground_truth_reference": "NASA LCROSS Impact Plume Spectrometry (2009)",\n      "mcda_suitability_score": 84.8,\n      "ai_confidence_pct": 90.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 90,\n      "temp_max_k": 210,\n      "diurnal_temperature_swing_k": 120\n    }\n  },\n  {\n    "id": "site-l",\n    "node_id": "12_Shoemaker_Crater_Rim",\n    "code": "Shoemaker",\n    "name": "Shoemaker Crater Rim South (Site L)",\n    "coordinates": {\n      "latitude": -88.1,\n      "longitude": 44.9,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-12-L"\n    },\n    "terrain_dem": {\n      "elevation_m": 3300,\n      "slope_deg": 5.1,\n      "roughness_rms_m": 0.9,\n      "crater_diameter_km": 50.9,\n      "rim_depth_m": 3500,\n      "landing_corridor_rating": "Gentle Southern Rim Shelf",\n      "accessibility_index_100": 86\n    },\n    "water_ice": {\n      "ice_probability_pct": 89.0,\n      "hydrogen_content_ppm": 1420,\n      "radar_cpr": 0.77,\n      "spectroscopy_band_3um_depth": 0.081,\n      "distance_to_psr_m": 390,\n      "estimated_ice_depth_m": 1.3,\n      "psr_name": "Shoemaker Deep Trap"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 85.2,\n      "max_continuous_light_days": 120,\n      "max_continuous_dark_days": 11.0,\n      "avg_solar_elevation_deg": 1.4,\n      "seasonal_variance_pct": 10.5\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 310,\n      "spe_hazard_tier": "Moderate",\n      "dose_rate_usv_h": 35.4,\n      "solar_cycle_phase": "Cycle 25 Maximum Modulation",\n      "terrain_shielding_factor_pct": 84.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Highland Regolith / Shock Breccia",\n      "crater_boundary": "Shoemaker / Malapert Ridge",\n      "earth_direct_los_pct": 93.6,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Near Side Polar Rim"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Autonomous Mining Rover Testing Grounds",\n      "mission_ground_truth_reference": "LRO LOLA / Diviner High-Resolution Grid",\n      "mcda_suitability_score": 86.5,\n      "ai_confidence_pct": 87.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 150,\n      "temp_max_k": 220,\n      "diurnal_temperature_swing_k": 70\n    }\n  },\n  {\n    "id": "ch3_shiv_shakti",\n    "node_id": "13_Chandrayaan3_Shiv_Shakti",\n    "code": "Shiv Shakti",\n    "name": "Chandrayaan-3 (Shiv Shakti Point)",\n    "coordinates": {\n      "latitude": -69.373,\n      "longitude": 32.319,\n      "hemisphere": "South Pole Region",\n      "grid_cell": "SP-13-M"\n    },\n    "terrain_dem": {\n      "elevation_m": -2580,\n      "slope_deg": 3.1,\n      "roughness_rms_m": 0.65,\n      "crater_diameter_km": 110.0,\n      "rim_depth_m": 2600,\n      "landing_corridor_rating": "Highland Inter-Crater Plains",\n      "accessibility_index_100": 94\n    },\n    "water_ice": {\n      "ice_probability_pct": 72.0,\n      "hydrogen_content_ppm": 620,\n      "radar_cpr": 0.45,\n      "spectroscopy_band_3um_depth": 0.038,\n      "distance_to_psr_m": 4200,\n      "estimated_ice_depth_m": 3.5,\n      "psr_name": "Manzinus C Shadow Pocket"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 80.0,\n      "max_continuous_light_days": 14,\n      "max_continuous_dark_days": 14.0,\n      "avg_solar_elevation_deg": 6.2,\n      "seasonal_variance_pct": 18.0\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 340,\n      "spe_hazard_tier": "Moderate",\n      "dose_rate_usv_h": 38.8,\n      "solar_cycle_phase": "Cycle 25 In-Situ Validated (ISRO ChaSTE)",\n      "terrain_shielding_factor_pct": 79.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Highland Regolith / Basaltic Breccia (Sulfur/Fe/Ti Rich)",\n      "crater_boundary": "Manzinus C / Simpelius N Plains",\n      "earth_direct_los_pct": 96.0,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Near Side South Polar Highland"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "In-Situ Ground Truth Soft Landing (First Polar Touchdown)",\n      "mission_ground_truth_reference": "ISRO Vikram Lander & Pragyan Rover (August 23, 2023)",\n      "mcda_suitability_score": 83.2,\n      "ai_confidence_pct": 96.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 130,\n      "temp_max_k": 333,\n      "diurnal_temperature_swing_k": 203\n    }\n  },\n  {\n    "id": "ch1_jawahar",\n    "node_id": "14_Chandrayaan1_Jawahar",\n    "code": "Jawahar Point",\n    "name": "Chandrayaan-1 (Jawahar Point)",\n    "coordinates": {\n      "latitude": -89.9,\n      "longitude": 0.0,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-14-N"\n    },\n    "terrain_dem": {\n      "elevation_m": 3950,\n      "slope_deg": 5.0,\n      "roughness_rms_m": 0.88,\n      "crater_diameter_km": 21.0,\n      "rim_depth_m": 4200,\n      "landing_corridor_rating": "Prime Polar Axis Rim",\n      "accessibility_index_100": 82\n    },\n    "water_ice": {\n      "ice_probability_pct": 92.0,\n      "hydrogen_content_ppm": 1820,\n      "radar_cpr": 0.86,\n      "spectroscopy_band_3um_depth": 0.095,\n      "distance_to_psr_m": 250,\n      "estimated_ice_depth_m": 1.0,\n      "psr_name": "Shackleton Polar Trap"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 91.5,\n      "max_continuous_light_days": 140,\n      "max_continuous_dark_days": 6.5,\n      "avg_solar_elevation_deg": 1.3,\n      "seasonal_variance_pct": 7.5\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 290,\n      "spe_hazard_tier": "Moderate",\n      "dose_rate_usv_h": 33.1,\n      "solar_cycle_phase": "Cycle 24 In-Situ Altimetry Reference",\n      "terrain_shielding_factor_pct": 83.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Highland Anorthosite Crust",\n      "crater_boundary": "Shackleton Zero Longitude Rim",\n      "earth_direct_los_pct": 97.0,\n      "relay_satellite_required": false,\n      "near_or_far_side": "South Polar Axis (0 deg Longitude)"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Historic Landmark of Water Molecule Confirmation (H2O/OH)",\n      "mission_ground_truth_reference": "ISRO MIP & Chandrayaan-1 M3 / CHACE (Nov 14, 2008)",\n      "mcda_suitability_score": 88.4,\n      "ai_confidence_pct": 93.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 160,\n      "temp_max_k": 220,\n      "diurnal_temperature_swing_k": 60\n    }\n  },\n  {\n    "id": "ch2_tiranga",\n    "node_id": "15_Chandrayaan2_Tiranga",\n    "code": "Tiranga Point",\n    "name": "Chandrayaan-2 (Tiranga Point)",\n    "coordinates": {\n      "latitude": -70.83,\n      "longitude": 22.68,\n      "hemisphere": "South Pole Region",\n      "grid_cell": "SP-15-O"\n    },\n    "terrain_dem": {\n      "elevation_m": -2300,\n      "slope_deg": 4.0,\n      "roughness_rms_m": 0.8,\n      "crater_diameter_km": 72.0,\n      "rim_depth_m": 2400,\n      "landing_corridor_rating": "Highland Shelf Plains",\n      "accessibility_index_100": 91\n    },\n    "water_ice": {\n      "ice_probability_pct": 74.0,\n      "hydrogen_content_ppm": 680,\n      "radar_cpr": 0.48,\n      "spectroscopy_band_3um_depth": 0.042,\n      "distance_to_psr_m": 3800,\n      "estimated_ice_depth_m": 3.0,\n      "psr_name": "Simpelius N Shadow Shelf"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 81.0,\n      "max_continuous_light_days": 14,\n      "max_continuous_dark_days": 14.0,\n      "avg_solar_elevation_deg": 5.8,\n      "seasonal_variance_pct": 17.5\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 335,\n      "spe_hazard_tier": "Moderate",\n      "dose_rate_usv_h": 38.2,\n      "solar_cycle_phase": "Cycle 25 Active DFSAR Radar Observations",\n      "terrain_shielding_factor_pct": 80.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Highland Regolith (0.32m Camera Surveyed)",\n      "crater_boundary": "Simpelius Highland Complex",\n      "earth_direct_los_pct": 95.0,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Near Side South Polar Highland"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "High-Resolution Dual-Frequency SAR Calibration Ground Truth",\n      "mission_ground_truth_reference": "ISRO Chandrayaan-2 Orbiter (OHRC 0.32m & DFSAR)",\n      "mcda_suitability_score": 81.9,\n      "ai_confidence_pct": 89.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 135,\n      "temp_max_k": 330,\n      "diurnal_temperature_swing_k": 195\n    }\n  },\n  {\n    "id": "lupex_ch4",\n    "node_id": "16_Chandrayaan4_LUPEX",\n    "code": "LUPEX / CH-4",\n    "name": "Chandrayaan-4 / LUPEX (Connecting Ridge Target)",\n    "coordinates": {\n      "latitude": -89.4,\n      "longitude": 145.0,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-16-P"\n    },\n    "terrain_dem": {\n      "elevation_m": 4050,\n      "slope_deg": 4.1,\n      "roughness_rms_m": 0.75,\n      "crater_diameter_km": 18.0,\n      "rim_depth_m": 2500,\n      "landing_corridor_rating": "Optimal Rover Drill Platform",\n      "accessibility_index_100": 90\n    },\n    "water_ice": {\n      "ice_probability_pct": 90.0,\n      "hydrogen_content_ppm": 1600,\n      "radar_cpr": 0.81,\n      "spectroscopy_band_3um_depth": 0.089,\n      "distance_to_psr_m": 320,\n      "estimated_ice_depth_m": 0.8,\n      "psr_name": "Connecting Ridge Southern Shadow Pockets"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 92.8,\n      "max_continuous_light_days": 145,\n      "max_continuous_dark_days": 5.5,\n      "avg_solar_elevation_deg": 1.4,\n      "seasonal_variance_pct": 6.8\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 285,\n      "spe_hazard_tier": "Moderate",\n      "dose_rate_usv_h": 32.5,\n      "solar_cycle_phase": "Cycle 25 Maximum Target Campaign",\n      "terrain_shielding_factor_pct": 86.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Cryogenic Subsurface Volatiles (1.5m Core Target)",\n      "crater_boundary": "Shackleton-de Gerlache High Ridge",\n      "earth_direct_los_pct": 97.5,\n      "relay_satellite_required": false,\n      "near_or_far_side": "South Polar Axis"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Joint ISRO-JAXA 1.5m Cryogenic Sample Return Target",\n      "mission_ground_truth_reference": "ISRO Heavy Lander + JAXA 350kg Cryo-Rover",\n      "mcda_suitability_score": 91.2,\n      "ai_confidence_pct": 92.0,\n      "suitability_tier": "HIGHLY SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 170,\n      "temp_max_k": 225,\n      "diurnal_temperature_swing_k": 55\n    }\n  },\n  {\n    "id": "apollo_11",\n    "node_id": "17_Apollo11_Tranquility_Base",\n    "code": "Apollo 11",\n    "name": "Apollo 11 (Statio Tranquillitatis)",\n    "coordinates": {\n      "latitude": 0.674,\n      "longitude": 23.473,\n      "hemisphere": "Equatorial Near Side",\n      "grid_cell": "EQ-17-Q"\n    },\n    "terrain_dem": {\n      "elevation_m": -1440,\n      "slope_deg": 1.8,\n      "roughness_rms_m": 0.4,\n      "crater_diameter_km": 800.0,\n      "rim_depth_m": 1500,\n      "landing_corridor_rating": "Ultra-Flat Titanium Basalt Plain",\n      "accessibility_index_100": 98\n    },\n    "water_ice": {\n      "ice_probability_pct": 20.0,\n      "hydrogen_content_ppm": 50,\n      "radar_cpr": 0.22,\n      "spectroscopy_band_3um_depth": 0.005,\n      "distance_to_psr_m": 99000,\n      "estimated_ice_depth_m": 0.0,\n      "psr_name": "None (Equatorial Zone)"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 50.0,\n      "max_continuous_light_days": 14,\n      "max_continuous_dark_days": 14.0,\n      "avg_solar_elevation_deg": 89.2,\n      "seasonal_variance_pct": 0.5\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 380,\n      "spe_hazard_tier": "High (Zero Polar Shadowing)",\n      "dose_rate_usv_h": 43.4,\n      "solar_cycle_phase": "Historic Solar Cycle 20 In-Situ Benchmarks",\n      "terrain_shielding_factor_pct": 72.0\n    },\n    "geographic_communications": {\n      "geological_unit": "High-Ti Basalt Flow (Mare Tranquillitatis)",\n      "crater_boundary": "Tranquility Basin Floor",\n      "earth_direct_los_pct": 100.0,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Prime Meridian Near Side"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Historic First Human Landing / Active LRRR Laser Ground Truth",\n      "mission_ground_truth_reference": "NASA Apollo 11 (July 20, 1969 - Armstrong & Aldrin)",\n      "mcda_suitability_score": 81.5,\n      "ai_confidence_pct": 98.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 100,\n      "temp_max_k": 385,\n      "diurnal_temperature_swing_k": 285\n    }\n  },\n  {\n    "id": "apollo_12",\n    "node_id": "18_Apollo12_Ocean_of_Storms",\n    "code": "Apollo 12",\n    "name": "Apollo 12 (Ocean of Storms / Surveyor 3)",\n    "coordinates": {\n      "latitude": -3.012,\n      "longitude": -23.422,\n      "hemisphere": "Equatorial Near Side",\n      "grid_cell": "EQ-18-R"\n    },\n    "terrain_dem": {\n      "elevation_m": -1520,\n      "slope_deg": 2.1,\n      "roughness_rms_m": 0.45,\n      "crater_diameter_km": 200.0,\n      "rim_depth_m": 1200,\n      "landing_corridor_rating": "Smooth Basalt Regolith",\n      "accessibility_index_100": 97\n    },\n    "water_ice": {\n      "ice_probability_pct": 20.0,\n      "hydrogen_content_ppm": 55,\n      "radar_cpr": 0.24,\n      "spectroscopy_band_3um_depth": 0.006,\n      "distance_to_psr_m": 99000,\n      "estimated_ice_depth_m": 0.0,\n      "psr_name": "None (Equatorial Zone)"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 50.0,\n      "max_continuous_light_days": 14,\n      "max_continuous_dark_days": 14.0,\n      "avg_solar_elevation_deg": 86.9,\n      "seasonal_variance_pct": 0.6\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 380,\n      "spe_hazard_tier": "High",\n      "dose_rate_usv_h": 43.4,\n      "solar_cycle_phase": "Historic Solar Cycle 20 Reference",\n      "terrain_shielding_factor_pct": 72.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Oceanus Procellarum Low-Ti Basalt",\n      "crater_boundary": "Surveyor Crater Margin",\n      "earth_direct_los_pct": 100.0,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Western Near Side"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Pinpoint Precision Touchdown Ground Truth (160m from Surveyor 3)",\n      "mission_ground_truth_reference": "NASA Apollo 12 (Nov 19, 1969 - Conrad & Bean)",\n      "mcda_suitability_score": 80.8,\n      "ai_confidence_pct": 97.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 100,\n      "temp_max_k": 387,\n      "diurnal_temperature_swing_k": 287\n    }\n  },\n  {\n    "id": "apollo_14",\n    "node_id": "19_Apollo14_Fra_Mauro",\n    "code": "Apollo 14",\n    "name": "Apollo 14 (Fra Mauro Highlands)",\n    "coordinates": {\n      "latitude": -3.645,\n      "longitude": -17.471,\n      "hemisphere": "Equatorial Near Side",\n      "grid_cell": "EQ-19-S"\n    },\n    "terrain_dem": {\n      "elevation_m": -1100,\n      "slope_deg": 3.4,\n      "roughness_rms_m": 0.68,\n      "crater_diameter_km": 95.0,\n      "rim_depth_m": 1800,\n      "landing_corridor_rating": "Highland Ridge Corridor",\n      "accessibility_index_100": 94\n    },\n    "water_ice": {\n      "ice_probability_pct": 22.0,\n      "hydrogen_content_ppm": 65,\n      "radar_cpr": 0.28,\n      "spectroscopy_band_3um_depth": 0.008,\n      "distance_to_psr_m": 99000,\n      "estimated_ice_depth_m": 0.0,\n      "psr_name": "None (Equatorial Zone)"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 50.0,\n      "max_continuous_light_days": 14,\n      "max_continuous_dark_days": 14.0,\n      "avg_solar_elevation_deg": 86.3,\n      "seasonal_variance_pct": 0.7\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 375,\n      "spe_hazard_tier": "High",\n      "dose_rate_usv_h": 42.8,\n      "solar_cycle_phase": "Historic Solar Cycle 20 Reference",\n      "terrain_shielding_factor_pct": 73.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Fra Mauro Formation (Imbrium Basin Impact Ejecta)",\n      "crater_boundary": "Cone Crater Flank",\n      "earth_direct_los_pct": 100.0,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Central Near Side"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Deep Impact Chronology & Basin Ejecta Ground Truth",\n      "mission_ground_truth_reference": "NASA Apollo 14 (Feb 5, 1971 - Shepard & Mitchell)",\n      "mcda_suitability_score": 81.2,\n      "ai_confidence_pct": 96.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 102,\n      "temp_max_k": 384,\n      "diurnal_temperature_swing_k": 282\n    }\n  },\n  {\n    "id": "apollo_15",\n    "node_id": "20_Apollo15_Hadley_Apennine",\n    "code": "Apollo 15",\n    "name": "Apollo 15 (Hadley-Apennine)",\n    "coordinates": {\n      "latitude": 26.132,\n      "longitude": 3.634,\n      "hemisphere": "Northern Near Side",\n      "grid_cell": "NN-20-T"\n    },\n    "terrain_dem": {\n      "elevation_m": -1800,\n      "slope_deg": 4.8,\n      "roughness_rms_m": 0.92,\n      "crater_diameter_km": 1100.0,\n      "rim_depth_m": 4500,\n      "landing_corridor_rating": "Apennine Mountain Plain",\n      "accessibility_index_100": 92\n    },\n    "water_ice": {\n      "ice_probability_pct": 22.0,\n      "hydrogen_content_ppm": 70,\n      "radar_cpr": 0.32,\n      "spectroscopy_band_3um_depth": 0.009,\n      "distance_to_psr_m": 99000,\n      "estimated_ice_depth_m": 0.0,\n      "psr_name": "None (Northern Mid-Latitude)"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 50.0,\n      "max_continuous_light_days": 14,\n      "max_continuous_dark_days": 14.0,\n      "avg_solar_elevation_deg": 63.8,\n      "seasonal_variance_pct": 1.5\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 370,\n      "spe_hazard_tier": "High",\n      "dose_rate_usv_h": 42.2,\n      "solar_cycle_phase": "Historic Solar Cycle 20 Reference",\n      "terrain_shielding_factor_pct": 74.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Anorthosite Crust (4.1 Gyr Genesis Rock) & Basalt",\n      "crater_boundary": "Montes Apenninus / Hadley Rille Gorge",\n      "earth_direct_los_pct": 100.0,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Northern Near Side"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "First Lunar Roving Vehicle (LRV) & Deep Crustal Ground Truth",\n      "mission_ground_truth_reference": "NASA Apollo 15 (July 30, 1971 - Scott & Irwin)",\n      "mcda_suitability_score": 82.4,\n      "ai_confidence_pct": 97.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 100,\n      "temp_max_k": 382,\n      "diurnal_temperature_swing_k": 282\n    }\n  },\n  {\n    "id": "apollo_16",\n    "node_id": "21_Apollo16_Descartes_Highlands",\n    "code": "Apollo 16",\n    "name": "Apollo 16 (Descartes Highlands)",\n    "coordinates": {\n      "latitude": -8.973,\n      "longitude": 15.498,\n      "hemisphere": "Central Near Side",\n      "grid_cell": "EQ-21-U"\n    },\n    "terrain_dem": {\n      "elevation_m": 1400,\n      "slope_deg": 4.2,\n      "roughness_rms_m": 0.82,\n      "crater_diameter_km": 48.0,\n      "rim_depth_m": 2800,\n      "landing_corridor_rating": "Central Highland Plateau",\n      "accessibility_index_100": 93\n    },\n    "water_ice": {\n      "ice_probability_pct": 22.0,\n      "hydrogen_content_ppm": 60,\n      "radar_cpr": 0.3,\n      "spectroscopy_band_3um_depth": 0.007,\n      "distance_to_psr_m": 99000,\n      "estimated_ice_depth_m": 0.0,\n      "psr_name": "None (Equatorial Highlands)"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 50.0,\n      "max_continuous_light_days": 14,\n      "max_continuous_dark_days": 14.0,\n      "avg_solar_elevation_deg": 81.0,\n      "seasonal_variance_pct": 0.8\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 365,\n      "spe_hazard_tier": "High",\n      "dose_rate_usv_h": 41.7,\n      "solar_cycle_phase": "Historic Solar Cycle 20 Reference",\n      "terrain_shielding_factor_pct": 74.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Cayley Plains / Descartes Highland Anorthosite Breccia",\n      "crater_boundary": "Descartes Crater East Plain",\n      "earth_direct_los_pct": 100.0,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Central Near Side"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Highland Impact Genesis Ground Truth (Proved Non-Volcanic)",\n      "mission_ground_truth_reference": "NASA Apollo 16 (April 21, 1972 - Young & Duke)",\n      "mcda_suitability_score": 82.0,\n      "ai_confidence_pct": 96.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 105,\n      "temp_max_k": 383,\n      "diurnal_temperature_swing_k": 278\n    }\n  },\n  {\n    "id": "apollo_17",\n    "node_id": "22_Apollo17_Taurus_Littrow",\n    "code": "Apollo 17",\n    "name": "Apollo 17 (Taurus-Littrow Valley)",\n    "coordinates": {\n      "latitude": 20.191,\n      "longitude": 30.772,\n      "hemisphere": "Northern Near Side",\n      "grid_cell": "NN-22-V"\n    },\n    "terrain_dem": {\n      "elevation_m": -2500,\n      "slope_deg": 3.9,\n      "roughness_rms_m": 0.78,\n      "crater_diameter_km": 750.0,\n      "rim_depth_m": 2500,\n      "landing_corridor_rating": "Mountain Enclosed Valley",\n      "accessibility_index_100": 95\n    },\n    "water_ice": {\n      "ice_probability_pct": 24.0,\n      "hydrogen_content_ppm": 85,\n      "radar_cpr": 0.35,\n      "spectroscopy_band_3um_depth": 0.011,\n      "distance_to_psr_m": 99000,\n      "estimated_ice_depth_m": 0.0,\n      "psr_name": "None (Pyroclastic Deposit Zone)"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 50.0,\n      "max_continuous_light_days": 14,\n      "max_continuous_dark_days": 14.0,\n      "avg_solar_elevation_deg": 69.8,\n      "seasonal_variance_pct": 1.4\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 370,\n      "spe_hazard_tier": "High",\n      "dose_rate_usv_h": 42.2,\n      "solar_cycle_phase": "Historic Solar Cycle 20 Reference",\n      "terrain_shielding_factor_pct": 75.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Pyroclastic Orange Volcanic Glass Beads & Ilmenite Basalt",\n      "crater_boundary": "Taurus-Littrow Valley / Shorty Crater",\n      "earth_direct_los_pct": 100.0,\n      "relay_satellite_required": false,\n      "near_or_far_side": "Northern Near Side"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "Deep Explosive Lunar Volcanism Ground Truth (110.5 kg Samples)",\n      "mission_ground_truth_reference": "NASA Apollo 17 (Dec 11, 1972 - Cernan & Schmitt)",\n      "mcda_suitability_score": 83.0,\n      "ai_confidence_pct": 98.0,\n      "suitability_tier": "SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 100,\n      "temp_max_k": 384,\n      "diurnal_temperature_swing_k": 284\n    }\n  },\n  {\n    "id": "artemis_3",\n    "node_id": "23_Artemis3_Ridge_Target",\n    "code": "Artemis III",\n    "name": "Artemis III Target (Shackleton-Malapert Ridge)",\n    "coordinates": {\n      "latitude": -89.5,\n      "longitude": 130.0,\n      "hemisphere": "South Pole",\n      "grid_cell": "SP-23-W"\n    },\n    "terrain_dem": {\n      "elevation_m": 4200,\n      "slope_deg": 3.7,\n      "roughness_rms_m": 0.68,\n      "crater_diameter_km": 25.0,\n      "rim_depth_m": 4300,\n      "landing_corridor_rating": "Optimal Starship HLS Corridor",\n      "accessibility_index_100": 91\n    },\n    "water_ice": {\n      "ice_probability_pct": 92.0,\n      "hydrogen_content_ppm": 1720,\n      "radar_cpr": 0.84,\n      "spectroscopy_band_3um_depth": 0.091,\n      "distance_to_psr_m": 310,\n      "estimated_ice_depth_m": 0.9,\n      "psr_name": "Shackleton Connecting Shadow Depressions"\n    },\n    "solar_illumination": {\n      "annual_sunlight_pct": 94.5,\n      "max_continuous_light_days": 175,\n      "max_continuous_dark_days": 4.0,\n      "avg_solar_elevation_deg": 1.42,\n      "seasonal_variance_pct": 5.2\n    },\n    "radiation_environment": {\n      "gcr_dose_msv_yr": 280,\n      "spe_hazard_tier": "Moderate",\n      "dose_rate_usv_h": 32.0,\n      "solar_cycle_phase": "Cycle 25 Maximum Modulation",\n      "terrain_shielding_factor_pct": 87.0\n    },\n    "geographic_communications": {\n      "geological_unit": "Polar Ridge Regolith / High Sintering Potential",\n      "crater_boundary": "Shackleton-Malapert Saddle",\n      "earth_direct_los_pct": 98.0,\n      "relay_satellite_required": false,\n      "near_or_far_side": "South Polar Axis"\n    },\n    "ai_ml_matrix": {\n      "ground_truth_label": "NASA Artemis III Crewed Landing Target (SpaceX Starship HLS)",\n      "mission_ground_truth_reference": "NASA Artemis Program Science Plan (2025+)",\n      "mcda_suitability_score": 93.8,\n      "ai_confidence_pct": 94.0,\n      "suitability_tier": "HIGHLY SUITABLE"\n    },\n    "environmental_temperatures": {\n      "temp_min_k": 175,\n      "temp_max_k": 222,\n      "diurnal_temperature_swing_k": 47\n    }\n  }\n]'

raw_sites = json.loads(LUNAR_DATASET_JSON)
print(f'✅ Loaded {len(raw_sites)} real lunar sites directly from embedded dataset')

def flatten_site(site):
    terrain = site.get('terrain_dem', {})
    ice     = site.get('water_ice', {})
    solar   = site.get('solar_illumination', {})
    rad     = site.get('radiation_environment', {})
    comms   = site.get('geographic_communications', {})
    ai      = site.get('ai_ml_matrix', {})
    temp    = site.get('environmental_temperatures', {})
    coords  = site.get('coordinates', {})
    return {
        'id':                        site.get('id'),
        'name':                      site.get('name'),
        'node_id':                   site.get('node_id'),
        'code':                      site.get('code'),
        'latitude':                  coords.get('latitude', 0),
        'longitude':                 coords.get('longitude', 0),
        'elevation_m':               terrain.get('elevation_m', 0),
        'slope_deg':                 terrain.get('slope_deg', 0),
        'roughness_rms_m':           terrain.get('roughness_rms_m', 0),
        'accessibility_index':       terrain.get('accessibility_index_100', 0),
        'ice_probability_pct':       ice.get('ice_probability_pct', 0),
        'hydrogen_ppm':              ice.get('hydrogen_content_ppm', 0),
        'radar_cpr':                 ice.get('radar_cpr', 0),
        'distance_to_psr_m':         ice.get('distance_to_psr_m', 0),
        'estimated_ice_depth_m':     ice.get('estimated_ice_depth_m', 0),
        'annual_sunlight_pct':       solar.get('annual_sunlight_pct', 0),
        'max_continuous_light_days': solar.get('max_continuous_light_days', 0),
        'max_continuous_dark_days':  solar.get('max_continuous_dark_days', 0),
        'avg_solar_elevation_deg':   solar.get('avg_solar_elevation_deg', 0),
        'seasonal_variance_pct':     solar.get('seasonal_variance_pct', 0),
        'gcr_dose_msv_yr':           rad.get('gcr_dose_msv_yr', 0),
        'dose_rate_usv_h':           rad.get('dose_rate_usv_h', 0),
        'terrain_shielding_pct':     rad.get('terrain_shielding_factor_pct', 0),
        'earth_los_pct':             comms.get('earth_direct_los_pct', 0),
        'relay_required':            1 if comms.get('relay_satellite_required', False) else 0,
        'temp_min_k':                temp.get('temp_min_k', 0),
        'temp_max_k':                temp.get('temp_max_k', 0),
        'diurnal_swing_k':           temp.get('diurnal_temperature_swing_k', 0),
        'mcda_suitability_score':    ai.get('mcda_suitability_score', 0),
        'ai_confidence_pct':         ai.get('ai_confidence_pct', 0),
        'suitability_tier':          ai.get('suitability_tier', 'MODERATE'),
    }

real_df = pd.DataFrame([flatten_site(s) for s in raw_sites])
real_df['dem_elevation_m'] = real_df['elevation_m']
print(f'✅ Feature matrix ready: {real_df.shape[0]} sites x {real_df.shape[1]} columns')
real_df[['name', 'mcda_suitability_score', 'suitability_tier']].head(10)


In [ ]:
# ─── Run this cell AFTER loading raw_sites above ────────────────────────────
# If you used Option A, raw_sites is already set.
# If you used Option C, parse the inline JSON:
# raw_sites = json.loads(RAW_SITES_JSON)

def flatten_site(site):
    """Flatten nested JSON into a single-level feature dict."""
    terrain = site.get('terrain_dem', {})
    ice     = site.get('water_ice', {})
    solar   = site.get('solar_illumination', {})
    rad     = site.get('radiation_environment', {})
    comms   = site.get('geographic_communications', {})
    ai      = site.get('ai_ml_matrix', {})
    temp    = site.get('environmental_temperatures', {})
    coords  = site.get('coordinates', {})

    return {
        # Identifiers
        'id':                        site.get('id'),
        'name':                      site.get('name'),
        'node_id':                   site.get('node_id'),
        'code':                      site.get('code'),
        'latitude':                  coords.get('latitude', 0),
        'longitude':                 coords.get('longitude', 0),

        # ── FEATURES (model inputs) ──────────────────────────────────────────
        # Terrain
        'elevation_m':               terrain.get('elevation_m', 0),
        'slope_deg':                 terrain.get('slope_deg', 0),
        'roughness_rms_m':           terrain.get('roughness_rms_m', 0),
        'accessibility_index':       terrain.get('accessibility_index_100', 0),

        # Water Ice
        'ice_probability_pct':       ice.get('ice_probability_pct', 0),
        'hydrogen_ppm':              ice.get('hydrogen_content_ppm', 0),
        'radar_cpr':                 ice.get('radar_cpr', 0),
        'distance_to_psr_m':         ice.get('distance_to_psr_m', 0),
        'estimated_ice_depth_m':     ice.get('estimated_ice_depth_m', 0),

        # Solar
        'annual_sunlight_pct':       solar.get('annual_sunlight_pct', 0),
        'max_continuous_light_days': solar.get('max_continuous_light_days', 0),
        'max_continuous_dark_days':  solar.get('max_continuous_dark_days', 0),
        'avg_solar_elevation_deg':   solar.get('avg_solar_elevation_deg', 0),
        'seasonal_variance_pct':     solar.get('seasonal_variance_pct', 0),

        # Radiation
        'gcr_dose_msv_yr':           rad.get('gcr_dose_msv_yr', 0),
        'dose_rate_usv_h':           rad.get('dose_rate_usv_h', 0),
        'terrain_shielding_pct':     rad.get('terrain_shielding_factor_pct', 0),

        # Communications
        'earth_los_pct':             comms.get('earth_direct_los_pct', 0),
        'relay_required':            1 if comms.get('relay_satellite_required', False) else 0,

        # Temperature
        'temp_min_k':                temp.get('temp_min_k', 0),
        'temp_max_k':                temp.get('temp_max_k', 0),
        'diurnal_swing_k':           temp.get('diurnal_temperature_swing_k', 0),

        # ── TARGET (model output) ────────────────────────────────────────────
        'mcda_suitability_score':    ai.get('mcda_suitability_score', 0),
        'ai_confidence_pct':         ai.get('ai_confidence_pct', 0),
        'suitability_tier':          ai.get('suitability_tier', 'MODERATE'),
    }

real_df = pd.DataFrame([flatten_site(s) for s in raw_sites])
print(f'✅ Loaded {len(real_df)} real lunar sites')
print(f'   Features available: {real_df.shape[1]} columns')
real_df[['name', 'mcda_suitability_score', 'suitability_tier']].head(10)

## Step 3: (Optional) Sample the 8GB LOLA Global DEM
This cell downloads the 8GB DEM and extracts real elevation/slope data at the 23 site coordinates to enrich the dataset. **Skip if you want a faster run.**

In [ ]:
DOWNLOAD_DEM = False  # Set to True to download the 8GB DEM in Colab (takes ~5 mins)

if DOWNLOAD_DEM:
    import subprocess
    print('Downloading 8GB LOLA DEM (this takes ~5 minutes on Colab)...')
    subprocess.run([
        'wget', '-q', '--show-progress',
        'https://planetarymaps.usgs.gov/mosaic/Lunar_LRO_LOLA_Global_LDEM_118m_Mar2014.tif',
        '-O', 'LOLA_DEM.tif'
    ])
    print('✅ DEM downloaded!')

    import rasterio
    from rasterio.transform import rowcol

    dem_elevations = []
    with rasterio.open('LOLA_DEM.tif') as src:
        print(f'DEM shape: {src.shape}, CRS: {src.crs}')
        for _, row in real_df.iterrows():
            try:
                r, c = rowcol(src.transform, row['longitude'], row['latitude'])
                window = rasterio.windows.Window(max(0, c-2), max(0, r-2), 5, 5)
                data = src.read(1, window=window)
                dem_elevations.append(float(np.nanmean(data)))
            except Exception as e:
                dem_elevations.append(row['elevation_m'])

    real_df['dem_elevation_m'] = dem_elevations
    print('✅ DEM elevation data extracted for all 23 sites!')
else:
    real_df['dem_elevation_m'] = real_df['elevation_m']
    print('⏭ Skipped DEM download — using dataset elevation values instead.')

## Step 4: Build Training Dataset via Physics-Informed Synthetic Augmentation
We augment our 23 real sites into 15,000 synthetic data points based on real lunar physics distributions.

In [ ]:
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
N = 15000  # Number of synthetic training samples

def lunar_physics_suitability(row):
    """Physics-informed Habitat Suitability Index (HSI) formula."""
    # Terrain score (lower slope + lower roughness = better)
    T = max(0, 100 - (row['slope_deg'] * 6) - (row['roughness_rms_m'] * 10)) * 0.9 + row['accessibility_index'] * 0.1
    T = np.clip(T, 0, 100)

    # Water Ice score
    W = (row['ice_probability_pct'] * 0.6 +
         min(100, row['hydrogen_ppm'] / 25) * 0.2 +
         max(0, 100 - row['distance_to_psr_m'] / 100) * 0.2)
    W = np.clip(W, 0, 100)

    # Solar score
    S = (row['annual_sunlight_pct'] * 0.7 +
         max(0, 100 - row['max_continuous_dark_days'] * 5) * 0.2 +
         max(0, 100 - row['seasonal_variance_pct'] * 5) * 0.1)
    S = np.clip(S, 0, 100)

    # Radiation safety score (lower dose = better)
    R = (row['terrain_shielding_pct'] * 0.5 +
         max(0, 100 - row['gcr_dose_msv_yr'] / 5) * 0.3 +
         max(0, 100 - row['dose_rate_usv_h'] / 0.6) * 0.2)
    R = np.clip(R, 0, 100)

    # Comms + Temperature
    C = row['earth_los_pct'] * (1 - row['relay_required'] * 0.2)
    TMP = max(0, 100 - row['diurnal_swing_k'] * 0.8)

    # Final HSI: weighted sum
    hsi = (T * 0.22 + W * 0.25 + S * 0.22 + R * 0.18 + C * 0.08 + TMP * 0.05)
    return round(float(np.clip(hsi + np.random.normal(0, 1.5), 0, 100)), 2)

# Compute real site stats for realistic sampling bounds
stats = real_df.describe()

def sample_around(col, n, noise_factor=0.3):
    """Sample values from a distribution centered around real data."""
    vals = real_df[col].values
    mean, std = vals.mean(), max(vals.std() * noise_factor, vals.std() * 0.1)
    sampled = np.random.normal(mean, std, n)
    return np.clip(sampled, vals.min() * 0.7, vals.max() * 1.15)

synth = pd.DataFrame({
    'elevation_m':               sample_around('elevation_m', N),
    'slope_deg':                 np.clip(np.random.exponential(5, N), 0.5, 45),
    'roughness_rms_m':           np.clip(np.random.exponential(1, N), 0.1, 8),
    'accessibility_index':       np.clip(np.random.normal(65, 20, N), 10, 100),
    'ice_probability_pct':       np.clip(np.random.beta(2, 2, N) * 100, 0, 100),
    'hydrogen_ppm':              np.clip(np.random.exponential(800, N), 0, 5000),
    'radar_cpr':                 np.clip(np.random.beta(2, 3, N), 0.05, 1.0),
    'distance_to_psr_m':         np.clip(np.random.exponential(2000, N), 0, 15000),
    'estimated_ice_depth_m':     np.clip(np.random.exponential(1.2, N), 0, 10),
    'annual_sunlight_pct':       np.clip(np.random.beta(3, 1.5, N) * 100, 0, 100),
    'max_continuous_light_days': np.clip(np.random.exponential(80, N), 0, 180),
    'max_continuous_dark_days':  np.clip(np.random.exponential(8, N), 0.5, 60),
    'avg_solar_elevation_deg':   np.clip(np.random.normal(5, 8, N), -90, 90),
    'seasonal_variance_pct':     np.clip(np.random.exponential(12, N), 0, 60),
    'gcr_dose_msv_yr':           np.clip(np.random.normal(350, 100, N), 150, 700),
    'dose_rate_usv_h':           np.clip(np.random.normal(40, 12, N), 10, 90),
    'terrain_shielding_pct':     np.clip(np.random.normal(65, 20, N), 0, 100),
    'earth_los_pct':             np.clip(np.random.beta(3, 1, N) * 100, 0, 100),
    'relay_required':            np.random.choice([0, 1], N, p=[0.7, 0.3]),
    'temp_min_k':                np.clip(np.random.normal(180, 40, N), 25, 290),
    'temp_max_k':                np.clip(np.random.normal(300, 60, N), 150, 400),
    'diurnal_swing_k':           np.clip(np.random.normal(120, 50, N), 10, 290),
    'dem_elevation_m':           sample_around('dem_elevation_m', N),
})

# Compute HSI target for synthetic data
synth['mcda_suitability_score'] = synth.apply(lunar_physics_suitability, axis=1)

print(f'✅ Generated {N:,} synthetic training samples')
print(f'   Score range: {synth["mcda_suitability_score"].min():.1f} – {synth["mcda_suitability_score"].max():.1f}')
print(f'   Mean score: {synth["mcda_suitability_score"].mean():.1f}')
synth.head(3)

## Step 5: Train the XGBoost Model with Optuna Hyperparameter Tuning

In [ ]:
import xgboost as xgb
import optuna
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

FEATURE_COLS = [
    'elevation_m', 'slope_deg', 'roughness_rms_m', 'accessibility_index',
    'ice_probability_pct', 'hydrogen_ppm', 'radar_cpr', 'distance_to_psr_m', 'estimated_ice_depth_m',
    'annual_sunlight_pct', 'max_continuous_light_days', 'max_continuous_dark_days',
    'avg_solar_elevation_deg', 'seasonal_variance_pct',
    'gcr_dose_msv_yr', 'dose_rate_usv_h', 'terrain_shielding_pct',
    'earth_los_pct', 'relay_required',
    'temp_min_k', 'temp_max_k', 'diurnal_swing_k',
    'dem_elevation_m'
]
TARGET_COL = 'mcda_suitability_score'

X_train = synth[FEATURE_COLS].values
y_train = synth[TARGET_COL].values

# ── Optuna hyperparameter search ────────────────────────────────────────────
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 1000),
        'max_depth':         trial.suggest_int('max_depth', 3, 9),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.5, 5),
        'tree_method':       'hist',
        'device':            'cuda',   # Uses GPU in Colab T4
        'random_state':      42
    }
    model = xgb.XGBRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=5,
                              scoring='neg_mean_absolute_error', n_jobs=-1)
    return scores.mean()

print('🔬 Running Optuna hyperparameter search (50 trials)...')
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

best_params = study.best_params
best_params.update({'tree_method': 'hist', 'device': 'cuda', 'random_state': 42})
print(f'\n✅ Best params found: {best_params}')
print(f'   Best CV MAE: {-study.best_value:.4f}')

In [ ]:
# ── Train the final model ────────────────────────────────────────────────────
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

final_model = xgb.XGBRegressor(**best_params)
final_model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)

y_pred_val = final_model.predict(X_val)
r2  = r2_score(y_val, y_pred_val)
mae = mean_absolute_error(y_val, y_pred_val)

print(f'\n📊 Final Model Validation Metrics:')
print(f'   R² Score: {r2:.4f}  (1.0 = perfect)')
print(f'   MAE:      {mae:.4f} suitability points')

# Save the model
import joblib
joblib.dump(final_model, 'xgb_lunar_model.pkl')
print('\n✅ Model saved as xgb_lunar_model.pkl')

## Step 6: SHAP Explainability — Feature Importance

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(final_model)

# Compute SHAP values on a background sample for speed
background = shap.sample(X_train, 500, random_state=42)
shap_values_bg = explainer.shap_values(background)

plt.figure(figsize=(12, 7))
shap.summary_plot(shap_values_bg, background, feature_names=FEATURE_COLS,
                  plot_type='bar', show=False, max_display=15)
plt.title('🌙 SHAP Feature Importance — Lunar Habitat Suitability', fontsize=14)
plt.tight_layout()
plt.savefig('shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP plot saved as shap_feature_importance.png')

## Step 7: Run Inference on All 23 Real Lunar Sites

In [ ]:
# Prepare real site feature matrix
X_real = real_df[FEATURE_COLS].values

# Predict suitability scores
predicted_scores = final_model.predict(X_real)
predicted_scores = np.clip(predicted_scores, 0, 100)

# Compute per-site SHAP values
shap_per_site = explainer.shap_values(X_real)

def assign_tier(score):
    if score >= 85:   return 'HIGHLY SUITABLE'
    elif score >= 68: return 'SUITABLE'
    elif score >= 50: return 'MODERATE'
    else:             return 'POOR'

def compute_confidence(score, shap_row, original_score):
    """AI confidence: higher when model agrees closely with original MCDA score."""
    agreement = max(0, 100 - abs(score - original_score) * 1.5)
    shap_magnitude = np.std(shap_row)
    certainty = max(60, 100 - shap_magnitude * 2)
    return round(float(np.clip((agreement * 0.6 + certainty * 0.4), 55, 99)), 1)

# Build per-site SHAP dict (top 5 features per site)
def top_shap_features(shap_row, n=5):
    indices = np.argsort(np.abs(shap_row))[::-1][:n]
    return [
        {'feature': FEATURE_COLS[i], 'shap_value': round(float(shap_row[i]), 4)}
        for i in indices
    ]

predictions = []
for i, (_, row) in enumerate(real_df.iterrows()):
    pred_score = float(round(predicted_scores[i], 2))
    orig_score = float(row['mcda_suitability_score'])
    shap_row   = shap_per_site[i]

    # Compute sub-factor scores from SHAP contributions
    factor_groups = {
        'terrain':           ['slope_deg', 'roughness_rms_m', 'accessibility_index', 'elevation_m', 'dem_elevation_m'],
        'waterIce':          ['ice_probability_pct', 'hydrogen_ppm', 'radar_cpr', 'distance_to_psr_m', 'estimated_ice_depth_m'],
        'solarIllumination': ['annual_sunlight_pct', 'max_continuous_light_days', 'max_continuous_dark_days', 'avg_solar_elevation_deg', 'seasonal_variance_pct'],
        'radiationSafety':   ['gcr_dose_msv_yr', 'dose_rate_usv_h', 'terrain_shielding_pct'],
        'temperature':       ['temp_min_k', 'temp_max_k', 'diurnal_swing_k'],
        'accessibility':     ['earth_los_pct', 'relay_required', 'accessibility_index']
    }

    def factor_score(group_cols):
        idxs = [FEATURE_COLS.index(c) for c in group_cols if c in FEATURE_COLS]
        raw = pred_score + sum(shap_row[idx] for idx in idxs) * 0.5
        return int(np.clip(raw, 10, 100))

    predictions.append({
        'id':                  row['id'],
        'node_id':             row['node_id'],
        'name':                row['name'],
        'code':                row['code'],
        'ai_suitability_score':     pred_score,
        'original_mcda_score':      orig_score,
        'score_delta':              round(pred_score - orig_score, 2),
        'ai_confidence_pct':        compute_confidence(pred_score, shap_row, orig_score),
        'suitability_tier':         assign_tier(pred_score),
        'ai_rank':                  0,  # filled after sorting
        'factors': {
            'terrain':           factor_score(factor_groups['terrain']),
            'waterIce':          factor_score(factor_groups['waterIce']),
            'solarIllumination': factor_score(factor_groups['solarIllumination']),
            'radiationSafety':   factor_score(factor_groups['radiationSafety']),
            'temperature':       factor_score(factor_groups['temperature']),
            'accessibility':     factor_score(factor_groups['accessibility']),
        },
        'shap_top_features':   top_shap_features(shap_row),
        'model_r2':            round(r2, 4),
        'model_mae':           round(mae, 4),
        'model_version':       'xgb_lunar_v1.0',
    })

# Sort by predicted score and assign ranks
predictions.sort(key=lambda x: x['ai_suitability_score'], reverse=True)
for rank, p in enumerate(predictions, 1):
    p['ai_rank'] = rank

print('✅ Inference complete on all 23 real lunar sites!')
print(f'\n🏆 Top 5 Sites by AI Suitability Score:')
for p in predictions[:5]:
    print(f"  #{p['ai_rank']} {p['name'][:50]:50s} → {p['ai_suitability_score']:.1f} ({p['suitability_tier']})")

## Step 8: Export `ai_predictions.json`
This file gets downloaded to your PC and injected into your React frontend.

In [ ]:
import json
from datetime import datetime

output = {
    'metadata': {
        'generated_at':      datetime.utcnow().isoformat() + 'Z',
        'model_version':     'xgb_lunar_v1.0',
        'training_samples':  N,
        'model_r2':          round(r2, 4),
        'model_mae':         round(mae, 4),
        'best_optuna_params': best_params,
        'feature_columns':   FEATURE_COLS,
    },
    'predictions': predictions
}

with open('ai_predictions.json', 'w') as f:
    json.dump(output, f, indent=2)

print('✅ ai_predictions.json saved!')
print(f'   File size: {len(json.dumps(output)) / 1024:.1f} KB')

# ── Auto-download in Colab ───────────────────────────────────────────────────
from google.colab import files
files.download('ai_predictions.json')
files.download('shap_feature_importance.png')

print('\n📥 Files downloading to your PC...')
print('\n🎯 NEXT STEP: Place ai_predictions.json in your ml_pipeline/ folder')
print('   Then run: python ml_pipeline/apply_predictions.py')

## (Optional) Step 9: Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

names     = [p['name'][:30] for p in predictions]
ai_scores = [p['ai_suitability_score'] for p in predictions]
orig_scores = [p['original_mcda_score'] for p in predictions]
tiers     = [p['suitability_tier'] for p in predictions]

color_map = {
    'HIGHLY SUITABLE': '#10b981',
    'SUITABLE':        '#3b82f6',
    'MODERATE':        '#f59e0b',
    'POOR':            '#ef4444',
}
colors = [color_map.get(t, '#6b7280') for t in tiers]

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.patch.set_facecolor('#0f172a')

# Bar chart
ax = axes[0]
ax.set_facecolor('#1e293b')
bars = ax.barh(names[::-1], ai_scores[::-1], color=colors[::-1], alpha=0.9, height=0.7)
ax.set_xlabel('AI Suitability Score', color='white', fontsize=11)
ax.set_title('🌙 AI-Ranked Lunar Habitat Sites', color='white', fontsize=13, fontweight='bold')
ax.tick_params(colors='white')
ax.spines['bottom'].set_color('#475569')
ax.spines['left'].set_color('#475569')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlim(0, 105)
for bar, score in zip(bars, ai_scores[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}', va='center', color='white', fontsize=8)

# Scatter: AI vs MCDA
ax2 = axes[1]
ax2.set_facecolor('#1e293b')
sc = ax2.scatter(orig_scores, ai_scores, c=colors, s=80, zorder=3, alpha=0.9)
lims = [min(min(orig_scores), min(ai_scores)) - 2,
        max(max(orig_scores), max(ai_scores)) + 2]
ax2.plot(lims, lims, 'w--', alpha=0.4, lw=1.5, label='Perfect Agreement')
ax2.set_xlabel('Original MCDA Score', color='white', fontsize=11)
ax2.set_ylabel('XGBoost AI Score', color='white', fontsize=11)
ax2.set_title(f'AI vs MCDA Agreement (R²={r2:.3f})', color='white', fontsize=13, fontweight='bold')
ax2.tick_params(colors='white')
ax2.spines['bottom'].set_color('#475569')
ax2.spines['left'].set_color('#475569')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.legend(facecolor='#334155', edgecolor='#475569', labelcolor='white')

plt.tight_layout()
plt.savefig('lunar_ai_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Results visualization saved!')